# Directions 2, 3 & 6: CRPS Loss, Conformal Prediction, Ensemble

This notebook implements three research directions:

| Direction | Description |
|-----------|-------------|
| **2. CRPS/Pinball training** | Train with CRPS or Pinball loss instead of NLL — directly optimizes evaluation metrics |
| **3. Conformal prediction** | Post-process model outputs to obtain intervals with finite-sample coverage guarantees |
| **6. Ensemble** | Quantile averaging and stacking of top models from Notebook 8 |

**Prerequisite**: Run Notebook 8 first to populate `results/price_floor_comparison/` (needed for Direction 6).

In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.linear_model import Ridge

current_dir = Path.cwd()
project_root = str(current_dir) if (current_dir / 'config.py').exists() else str(current_dir.parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import DataConfig, ExperimentConfig, ModelConfig, TrainingConfig
from models import ProbabilisticTransformer, HybridProbabilisticTransformerOUJump
from core.experiment_utils import (
    load_data, load_cache, save_cache, run_experiment, load_run,
    QUANTILE_LEVELS, CANONICAL_MODEL_CONFIG, CANONICAL_TRAIN_CONFIG,
)
from core.trainer import Trainer
from core.evaluator import Evaluator

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

try:
    gpus = tf.config.experimental.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs: {len(gpus)}")
except Exception:
    pass

2026-03-10 17:35:56.548702: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773160556.564868  112020 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773160556.569792  112020 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773160556.582187  112020 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773160556.582202  112020 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773160556.582204  112020 computation_placer.cc:177] computation placer alr

GPUs: 1


In [2]:
RESULTS_DIR = Path(project_root) / "results" / "directions_210"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = RESULTS_DIR / "results.json"
PRICE_FLOOR_DIR = Path(project_root) / "results" / "price_floor_comparison"

data = load_data()

Data loaded  —  Train: 10366, Val: 1997, Test: 3722


---
## Direction 2: CRPS and Pinball Training Loss

Compare models trained with different losses:
- **Johnson SU (NLL)** — baseline, trains on negative log-likelihood
- **Gaussian CRPS** — trains on CRPS (closed-form for Gaussian)
- **Quantile (Pinball)** — trains on pinball loss for multiple quantiles

In [3]:
cache = load_cache(CACHE_FILE)

run_experiment(
    ProbabilisticTransformer, "Johnson SU (NLL)", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=False, head_type="johnson_su",
)

run_experiment(
    ProbabilisticTransformer, "Gaussian CRPS", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=False, head_type="gaussian_crps",
)

run_experiment(
    ProbabilisticTransformer, "Quantile (Pinball)", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=False,
    head_type="quantile",
    head_params={"quantiles": [0.025, 0.1, 0.25, 0.5, 0.75, 0.9, 0.975]},
)

save_cache(CACHE_FILE, cache)

[Johnson SU (NLL)] found in cache — skipping.
[Gaussian CRPS] found in cache — skipping.
[Quantile (Pinball)] found in cache — skipping.


In [4]:
df_d2 = pd.DataFrame([{"Model": k, **v} for k, v in cache.items() if "Johnson" in k or "Gaussian" in k or "Quantile" in k])
if not df_d2.empty:
    display(df_d2.sort_values("MAE"))

,Model,MAE,MSE,RMSE,MAPE,R2,PICP,MPIW,PINAW,IntervalScore,CRPS,NLL,Pinball_10,Pinball_50,Pinball_90,Avg_Pinball,training_time,radius
4,Quantile avg ensemble,18.556695,NaN,NaN,NaN,NaN,0.958109,103.617133,0.287660,128.896866,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Quantile (Pinball),19.908847,700.900569,26.467164,1856.585048,0.501794,0.902195,94.908981,0.263484,151.113886,14.659389,NaN,5.000447,9.954423,4.752274,6.569048,104.145579,NaN
0,Johnson SU (NLL),19.979567,715.701533,26.741236,1915.172410,0.491273,0.904622,93.926795,0.260757,145.900137,14.299280,53.976532,4.848948,9.686712,4.640926,6.392195,100.734397,NaN
1,Gaussian CRPS,20.724185,755.167333,27.458572,1776.574279,0.463221,0.834182,72.561564,0.201444,172.198488,15.323731,32963.283408,5.176506,10.362093,5.075456,6.871352,120.683194,NaN
3,Conformal (Johnson SU),22.035421,NaN,NaN,NaN,NaN,0.952758,124.183412,0.344755,156.160491,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.091706


---
## Direction 3: Conformal Prediction

Split conformal prediction: use validation residuals to calibrate symmetric intervals.
Guarantees marginal coverage ≥ 1−α (e.g. 95%) under exchangeability.

In [5]:
def conformal_calibrate(y_true: np.ndarray, y_pred: np.ndarray, alpha: float = 0.05) -> float:
    """Compute conformal radius from calibration residuals."""
    scores = np.abs(y_true - y_pred).flatten()
    n = len(scores)
    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_level = min(q_level, 1.0)
    return float(np.quantile(scores, q_level))


def apply_conformal_intervals(
    y_pred_mean: np.ndarray,
    radius: float,
) -> tuple:
    """Apply symmetric conformal intervals."""
    q_low = y_pred_mean - radius
    q_high = y_pred_mean + radius
    return q_low, q_high


def evaluate_intervals(
    y_test: np.ndarray,
    y_pred_mean: np.ndarray,
    q_low: np.ndarray,
    q_high: np.ndarray,
    alpha: float = 0.05,
) -> dict:
    """Compute interval metrics."""
    covered = (y_test >= q_low) & (y_test <= q_high)
    picp = float(np.mean(covered))
    mpiw = float(np.mean(q_high - q_low))
    y_range = np.max(y_test) - np.min(y_test)
    pinaw = float(mpiw / y_range) if y_range > 0 else np.nan
    width = q_high - q_low
    lower_pen = (2 / alpha) * (q_low - y_test) * (y_test < q_low)
    upper_pen = (2 / alpha) * (y_test - q_high) * (y_test > q_high)
    interval_score = float(np.mean(width + lower_pen + upper_pen))
    mae = float(np.mean(np.abs(y_test - y_pred_mean)))
    return {"PICP": picp, "MPIW": mpiw, "PINAW": pinaw, "IntervalScore": interval_score, "MAE": mae}

In [6]:
CONFORMAL_KEY = "Conformal (Johnson SU)"
cache = load_cache(CACHE_FILE)

if CONFORMAL_KEY in cache:
    metrics_cp = {k: v for k, v in cache[CONFORMAL_KEY].items() if k != "radius"}
    radius = cache[CONFORMAL_KEY].get("radius", np.nan)
    print(f"[{CONFORMAL_KEY}] found in cache — skipping.")
    print(f"\nConformal prediction (95% target coverage, radius={radius:.2f}):")
    for k, v in metrics_cp.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) and not np.isnan(v) else f"  {k}: {v}")
else:
    import gc

    tf.keras.backend.clear_session()
    gc.collect()
    np.random.seed(42)
    tf.random.set_seed(42)

    exp_conf = ExperimentConfig(
        name="conformal_baseline",
        data_config=data.data_config,
        model_config=ModelConfig(**CANONICAL_MODEL_CONFIG),
        training_config=TrainingConfig(**CANONICAL_TRAIN_CONFIG),
        head_type="johnson_su",
    )

    model = ProbabilisticTransformer(exp_conf)
    trainer = Trainer(exp_conf)
    print("Training baseline model for conformal calibration …")
    trainer.train(model, data.X_train_s, data.y_train_s, data.X_val_s, data.y_val_s, verbose=2)
    print("Training complete.\n")

    print("Generating validation & test forecasts …")
    evaluator = Evaluator(model, data.scaler)
    y_pred_val = evaluator.generate_forecasts(data.X_val_s)
    y_pred_test = evaluator.generate_forecasts(data.X_test_s)
    print(f"  val predictions: {y_pred_val.shape}, test predictions: {y_pred_test.shape}")

    alpha = 0.05
    radius = conformal_calibrate(data.y_val, y_pred_val, alpha)
    q_low_cp, q_high_cp = apply_conformal_intervals(y_pred_test, radius)

    metrics_cp = evaluate_intervals(data.y_test, y_pred_test, q_low_cp, q_high_cp, alpha)
    cache[CONFORMAL_KEY] = {**metrics_cp, "radius": radius}
    save_cache(CACHE_FILE, cache)
    print(f"\nConformal prediction (95% target coverage, radius={radius:.2f}):")
    for k, v in metrics_cp.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) and not np.isnan(v) else f"  {k}: {v}")

[Conformal (Johnson SU)] found in cache — skipping.

Conformal prediction (95% target coverage, radius=62.09):
  PICP: 0.9528
  MPIW: 124.1834
  PINAW: 0.3448
  IntervalScore: 156.1605
  MAE: 22.0354


---
## Direction 6: Ensemble (Quantile Averaging & Stacking)

Load predictions from top models in Notebook 8 and combine them:
- **Quantile averaging**: average predicted quantiles across models
- **Stacking**: ridge regression meta-learner on point forecasts (using val predictions)

In [7]:
def load_model_predictions(base_dir: Path, model_name: str, run_idx: int = 0) -> dict | None:
    """Load predictions from a saved run. model_name as in experiment (e.g. 'Baseline (Johnson SU)')."""
    subdir = model_name.replace(" ", "_")
    run_path = base_dir / subdir / f"run_{run_idx}" / "predictions.npz"
    if not run_path.exists():
        return None
    data = np.load(run_path)
    return {k: data[k] for k in data.files}


def quantile_average(preds_list: list[dict], quantile_keys: list[str]) -> dict:
    """Average quantile predictions across models."""
    out = {}
    for qk in quantile_keys:
        key = f"q_{qk}" if not qk.startswith("q_") else qk
        arrs = [p[key] for p in preds_list if key in p]
        if arrs:
            out[key] = np.mean(arrs, axis=0)
    if "y_pred_mean" in preds_list[0]:
        out["y_pred_mean"] = np.mean([p["y_pred_mean"] for p in preds_list], axis=0)
    return out


def evaluate_ensemble_predictions(preds: dict, y_test: np.ndarray, alpha: float = 0.05) -> dict:
    """Compute metrics for ensemble predictions."""
    y_pred = preds.get("y_pred_mean")
    q_low = preds.get("q_0.025")
    q_high = preds.get("q_0.975")
    if q_low is None:
        q_keys = [k for k in preds if k.startswith("q_")]
        q_low = preds.get(min(q_keys, key=lambda x: float(x.split("_")[1])), y_pred) if q_keys else y_pred
    if q_high is None:
        q_keys = [k for k in preds if k.startswith("q_")]
        q_high = preds.get(max(q_keys, key=lambda x: float(x.split("_")[1])), y_pred) if q_keys else y_pred
    return evaluate_intervals(y_test, y_pred, q_low, q_high, alpha)

In [8]:
# Top models from Notebook 8 (by MAE)
ENSEMBLE_MODELS = [
    "Hybrid + OU + Jump",
    "Baseline (Johnson SU)",
    "Johnson SU + Floor",
    "Hybrid + Hourly OU",
]

QUANTILE_AVG_KEY = "Quantile avg ensemble"
cache = load_cache(CACHE_FILE)

preds_list = []
for name in ENSEMBLE_MODELS:
    p = load_model_predictions(PRICE_FLOOR_DIR, name)
    if p is not None:
        preds_list.append(p)
        print(f"Loaded: {name}")

if len(preds_list) < 2:
    print("\nRun Notebook 8 first to populate results/price_floor_comparison/")
elif QUANTILE_AVG_KEY in cache:
    metrics_ens = {k: v for k, v in cache[QUANTILE_AVG_KEY].items()}
    print(f"\n[{QUANTILE_AVG_KEY}] found in cache — skipping.")
    print("\nQuantile-averaged ensemble:")
    for k, v in metrics_ens.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) and not np.isnan(v) else f"  {k}: {v}")
else:
    q_keys = [0.025, 0.1, 0.25, 0.5, 0.75, 0.9, 0.975]
    pred_keys = [f"q_{q}" for q in q_keys]
    preds_avg = quantile_average(preds_list, pred_keys)
    preds_avg["y_pred_mean"] = np.mean([p["y_pred_mean"] for p in preds_list], axis=0)
    y_test_ref = preds_list[0]["y_test"]
    metrics_ens = evaluate_ensemble_predictions(preds_avg, y_test_ref)
    cache[QUANTILE_AVG_KEY] = metrics_ens
    save_cache(CACHE_FILE, cache)
    print("\nQuantile-averaged ensemble:")
    for k, v in metrics_ens.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) and not np.isnan(v) else f"  {k}: {v}")

Loaded: Hybrid + OU + Jump
Loaded: Baseline (Johnson SU)
Loaded: Johnson SU + Floor
Loaded: Hybrid + Hourly OU

[Quantile avg ensemble] found in cache — skipping.

Quantile-averaged ensemble:
  PICP: 0.9581
  MPIW: 103.6171
  PINAW: 0.2877
  IntervalScore: 128.8969
  MAE: 18.5567


In [9]:
# Stacking: meta-learner on point forecasts (requires val predictions from each model)
# We train each model once and collect val preds, then fit Ridge on (val_preds -> val_true)
def run_stacking_ensemble(data, model_configs, n_runs=1):
    """Train base models, collect val preds, fit meta-learner, evaluate on test."""
    from core.experiment_utils import set_seeds
    val_preds_list = []
    test_preds_list = []
    for name, model_cls, head_type, head_params, is_hybrid in model_configs:
        tf.keras.backend.clear_session()
        set_seeds(0)
        exp_conf = ExperimentConfig(
            name=name,
            data_config=data.data_config,
            model_config=ModelConfig(**CANONICAL_MODEL_CONFIG),
            training_config=TrainingConfig(**CANONICAL_TRAIN_CONFIG),
            head_type=head_type,
            head_params=head_params or {},
        )
        model = model_cls(exp_conf)
        trainer = Trainer(exp_conf)
        trainer.train(model, data.X_train_s, data.y_train_s, data.X_val_s, data.y_val_s)
        if is_hybrid:
            model.fit_ou(data.X_train_s, data.y_train_s)
            val_pred = model.predict_hybrid(data.X_val_s)
            test_pred = model.predict_hybrid(data.X_test_s)
            _, val_pred = data.scaler.inverse_transform(X=None, y=val_pred)
            _, test_pred = data.scaler.inverse_transform(X=None, y=test_pred)
        else:
            ev = Evaluator(model, data.scaler)
            val_pred = ev.generate_forecasts(data.X_val_s)
            test_pred = ev.generate_forecasts(data.X_test_s)
        val_preds_list.append(val_pred.flatten())
        test_preds_list.append(test_pred.flatten())
    X_meta_val = np.column_stack(val_preds_list)
    X_meta_test = np.column_stack(test_preds_list)
    meta = Ridge(alpha=1.0).fit(X_meta_val, data.y_val.flatten())
    y_pred_stacked = meta.predict(X_meta_test).reshape(data.y_test.shape)
    mae = float(np.mean(np.abs(data.y_test - y_pred_stacked)))
    return mae, y_pred_stacked


STACKING_CONFIGS = [
    ("Baseline (Johnson SU)", ProbabilisticTransformer, "johnson_su", {}, False),
    ("Hybrid + OU + Jump", HybridProbabilisticTransformerOUJump, "johnson_su", {}, True),
]

STACKING_KEY = "Stacking (2 models)"
cache = load_cache(CACHE_FILE)

if STACKING_KEY in cache:
    mae_stacked = cache[STACKING_KEY]["MAE"]
    y_stacked = None  # not needed when loading from cache
    print(f"[{STACKING_KEY}] found in cache — skipping.")
    print(f"  MAE (stacked): {mae_stacked:.4f}")
else:
    print("Stacking (Ridge meta-learner on 2 base models)...")
    mae_stacked, y_stacked = run_stacking_ensemble(data, STACKING_CONFIGS)
    cache[STACKING_KEY] = {"MAE": mae_stacked}
    save_cache(CACHE_FILE, cache)
    print(f"  MAE (stacked): {mae_stacked:.4f}")

[Stacking (2 models)] found in cache — skipping.
  MAE (stacked): 22.2009


In [10]:
# Summary: compare all approaches (built from cache for reliable reload)
cache = load_cache(CACHE_FILE)

D2_KEYS = ["Johnson SU (NLL)", "Gaussian CRPS", "Quantile (Pinball)"]
summary_rows = []
for k in D2_KEYS:
    if k in cache:
        v = cache[k]
        summary_rows.append({"Approach": k, "MAE": v.get("MAE"), "PICP": v.get("PICP"), "Source": "D2"})
if "Conformal (Johnson SU)" in cache:
    v = cache["Conformal (Johnson SU)"]
    summary_rows.append({"Approach": "Conformal (Johnson SU)", "MAE": v.get("MAE"), "PICP": v.get("PICP"), "Source": "D3"})
if "Quantile avg ensemble" in cache:
    v = cache["Quantile avg ensemble"]
    summary_rows.append({"Approach": "Quantile avg ensemble", "MAE": v.get("MAE"), "PICP": v.get("PICP"), "Source": "D6"})
if "Stacking (2 models)" in cache:
    v = cache["Stacking (2 models)"]
    summary_rows.append({"Approach": "Stacking (2 models)", "MAE": v.get("MAE"), "PICP": np.nan, "Source": "D6"})

df_summary = pd.DataFrame(summary_rows)
if not df_summary.empty:
    display(df_summary)
    df_summary.to_csv(RESULTS_DIR / "comparison_summary.csv", index=False)

,Approach,MAE,PICP,Source
0,Johnson SU (NLL),19.979567,0.904622,D2
1,Gaussian CRPS,20.724185,0.834182,D2
2,Quantile (Pinball),19.908847,0.902195,D2
3,Conformal (Johnson SU),22.035421,0.952758,D3
4,Quantile avg ensemble,18.556695,0.958109,D6
5,Stacking (2 models),22.200889,NaN,D6
